## SBERT Cluster-Based Extractive Summarizer

**Objective:** Implement a transformer embedding-based extractive summarizer using Sentence-BERT (SBERT) and K-Means clustering, and evaluate it using standard ROUGE metrics. Also compute coverage and redundancy metrics across TF-IDF, LexRank and SBERT embedding methods.

**Dataset:** Same 500-article WikiHow sample used in LexRank notebook, loaded via saved CSV for consistency across notebooks.

**Steps:**
1. Load the dataset and restore the `sentences` column (list of tokenized sentences per article)
2. Encode each article's sentences into 384-dimensional embeddings using `all-MiniLM-L6-v2`
3. Cluster embeddings into 5 groups using K-Means
4. Select the sentence closest to each cluster centroid as a representative summary sentence
5. Evaluate against reference headlines using ROUGE-1, ROUGE-2, and ROUGE-L (F-measure)
6. Compute **coverage** (ROUGE-1 recall) for all three methods
7. Compute **redundancy** for all three methods — mean pairwise cosine similarity between summary sentences (via SBERT embeddings), using the upper triangle of the similarity matrix to avoid double-counting pairs
8. Save the final combined dataframe (all summaries, scores, coverage, redundancy) as a pickle file for the visualization notebook

**Evaluation:** ROUGE (F-measure), Coverage (ROUGE-1 recall), Redundancy (cosine similarity).

**Results (mean across 500 articles):**

| Method | ROUGE-1 | ROUGE-2 | ROUGE-L | Coverage | Redundancy |
|---|---|---|---|---|---|
| TF-IDF | 0.250 | 0.070 | 0.136 | 0.532 | 0.399 |
| LexRank | 0.278 | 0.078 | 0.152 | 0.487 | 0.442 |
| SBERT | 0.309 | 0.072 | 0.166 | 0.370 | 0.325 |

SBERT achieves the best overall ROUGE performance and lowest redundancy, but the lowest lexical coverage — likely reflecting ROUGE's lexical bias rather than genuinely poorer summarization quality.

In [11]:
from sentence_transformers import SentenceTransformer
import pandas as pd

#### Loading the Model:

In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
sample = pd.read_csv('sample1.csv')

In [14]:
sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   headline          500 non-null    str    
 1   title             500 non-null    str    
 2   text              500 non-null    str    
 3   word count        500 non-null    int64  
 4   sentences         500 non-null    str    
 5   tfidf_summary     500 non-null    str    
 6   tfidf_score       500 non-null    str    
 7   rouge1_f          500 non-null    float64
 8   rouge2_f          500 non-null    float64
 9   rougeL_f          500 non-null    float64
 10  lexrank_summary   500 non-null    str    
 11  lexrank_score     500 non-null    str    
 12  lexrank_rouge1_f  500 non-null    float64
 13  lexrank_rouge2_f  500 non-null    float64
 14  lexrank_rougeL_f  500 non-null    float64
dtypes: float64(6), int64(1), str(8)
memory usage: 9.0 MB


In [15]:
sample.head(3)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,lexrank_summary,lexrank_score,lexrank_rouge1_f,lexrank_rouge2_f,lexrank_rougeL_f
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,['Meditating is a great way to relax your mind...,"When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627,Going out to the movies or watching a movie on...,{'rouge1': Score(precision=0.08074534161490683...,0.144444,0.078652,0.144444
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"['The beauty industry is a huge one, and it’s ...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954,Before you begin your career as a beauty exper...,{'rouge1': Score(precision=0.17355371900826447...,0.234637,0.022599,0.134078
2,Relax. Make your move before you psyche yourse...,How to Ask for a Phone Number,If there's one single thing you can do to make...,2415,"[""If there's one single thing you can do to ma...",Though it's always difficult (some might say a...,{'rouge1': Score(precision=0.20652173913043478...,0.301587,0.058511,0.153439,If there's one single thing you can do to make...,"{'rouge1': Score(precision=0.3299492385786802,...",0.434783,0.161616,0.260870


In [16]:
sample['sentences']
# After saving and loading from CSV, dtype of sentences has changed to 'str' from 'object'. We need to convert it back to object.

0      ['Meditating is a great way to relax your mind...
1      ['The beauty industry is a huge one, and it’s ...
2      ["If there's one single thing you can do to ma...
3      ['You are eligible to buy a hunting license so...
4      ["Other people are outside your control, and i...
                             ...                        
495    ['If you want to build abdominal strength and ...
496    ['If you have not bought shoes for a while, co...
497    ['Warm water will open up your pores and help ...
498    ['Leafy greens are an integral part of a raw d...
499    ['In most cases, you must have a law degree to...
Name: sentences, Length: 500, dtype: str

In [17]:
import ast

In [18]:
sample['sentences'] = sample['sentences'].apply(ast.literal_eval)

In [19]:
sample['sentences'] # dtype of sentences is now back to object.

0      [Meditating is a great way to relax your mind,...
1      [The beauty industry is a huge one, and it’s e...
2      [If there's one single thing you can do to mak...
3      [You are eligible to buy a hunting license so ...
4      [Other people are outside your control, and if...
                             ...                        
495    [If you want to build abdominal strength and f...
496    [If you have not bought shoes for a while, con...
497    [Warm water will open up your pores and help y...
498    [Leafy greens are an integral part of a raw di...
499    [In most cases, you must have a law degree to ...
Name: sentences, Length: 500, dtype: object

In [20]:
type(sample['sentences'].iloc[0])

list

### Training the Model:<br>(Generating vector embeddings of each sentences)


In [21]:
len(sample['sentences'].iloc[0])

51

In [22]:
sentences = sample['sentences'].iloc[0]
sentences

['Meditating is a great way to relax your mind, and you can meditate almost anywhere and at any time.',
 'Just pick a quiet place where you can sit on level ground and close your eyes.',
 'Cross your legs and keep your hands on your lap.',
 'Focus on inhaling and exhaling, and let your body be governed by your breath.',
 'Keep as still as possible and avoid fidgeting.',
 "Be aware of what you can't control.",
 'Focus and absorb the smells and sounds around you.',
 'Clear your mind.',
 "Don't think about how much work you have left to do, or about what you're going to make for dinner.",
 'Just focus on clearing your mind and managing your breath.',
 'Relax every part of your body.',
 'You can focus on one part of your body at a time until you feel that every part of you is loose and relaxed.',
 'Going out to the movies or watching a movie on television can help you escape into another universe and to take your mind off of your own problems.',
 "When you watch a movie, try to clear your 

In [23]:
embeddings = model.encode(sentences)
embeddings.shape

(51, 384)

(no. of rows, no. of columns)  
(sentences, dimensional vectors) : (51, 384) ... i.e. 1st article{sample['sentences'].iloc[0]} has 51 sentences.   
Note:Each sentence has 384 dimensional vectors.

There are as many dimensions as there are variables i.e. columns  --> Correct in older TF-IDF methods not in BERT.  

[Q. why 384 dimensions here though? columns in the df are definitely less than that...check!]  
Ans: Each sentence vector will have as many dimensions(384 in this case) as there are unique words in the entire article/document. -->This is true in old TF-IDF methods but sentence embeddings models don't work that way.  

-->Correct Answer: 384 dimension is a fixed number baked into pre-trained model like 'all-MiniLM-L6-v2'. These models are designed to output exactly 384-dimensional vectors irrespective of input size. So there could be 1, 5, 5000 sentences, but the model would always ouput 384 dimesional vectors for each of those sentences.

In [24]:
embeddings

array([[ 0.11764733,  0.00411615,  0.03602198, ...,  0.09834801,
        -0.11119489,  0.03668928],
       [ 0.1144508 , -0.15487695,  0.06278692, ...,  0.07402904,
        -0.15181278, -0.01350368],
       [-0.02087728, -0.03308865, -0.00659909, ...,  0.03758969,
        -0.07640935,  0.0574163 ],
       ...,
       [ 0.12094411, -0.01500227, -0.02703045, ..., -0.04808583,
        -0.11497065,  0.02931774],
       [ 0.03235476, -0.09572972,  0.02300628, ..., -0.00662886,
        -0.00248602, -0.01293685],
       [ 0.09465639, -0.01586493,  0.05361329, ..., -0.01697778,
        -0.05289493, -0.02875265]], shape=(51, 384), dtype=float32)

### Intializing KMeans: <br> 
Make 5 Clusters of the generated embeddings

In [25]:
from sklearn.cluster import KMeans

Understanding Kmeans:  
1. K-Means groups sentences  into N clusters and finds the geometric center of each cluster.
2. That center is a O-dimensional vector — the "average" position of all sentences in that cluster.  
[These centre vectors are stored in **'kmeans.cluster_centers_'** ] 
3. Then you find which actual sentence is closest to each center - These best represents those clusters and makes the summary.

--> **pairwise_distances_argmin()** - Used to find sentence closest to cluster centre.

Note:  
A point in 2D space needs 2 numbers: (x, y)  
A point in 3D space needs 3 numbers: (x, y, z)  
A point in 384D space needs 384 numbers: (x1, x2, x3, ... x384)  
So each row in kmeans.cluster_centers_ is just one point — but in a very high dimensional space. Those 384 numbers together define one location in that space.  
[Like in 2D, even a single point can make up a vector(position vector), just like that in 384D, 384 points make up 1 (centre)vector.]

In [26]:
kmeans = KMeans(n_clusters=5, random_state=0, n_init='auto').fit(embeddings) 
# n_clusters : Should be less than total no. of sentences present in an article.

In [27]:
kmeans.cluster_centers_ 
# kmeans.cluster_centers_ gives centre vector per cluster.(So, 5 cluster centre would mean it would give 5 centre vector, one for each cluster.)
# So below we have 5 centre vector, each row representing one of them.

array([[ 0.05730813,  0.00237473,  0.00875747, ...,  0.04597862,
        -0.07220785,  0.01590482],
       [ 0.09174717, -0.05323385,  0.02045238, ...,  0.01461591,
        -0.07556296,  0.0034425 ],
       [ 0.05868079, -0.04335779,  0.03348997, ...,  0.03736767,
        -0.09241115,  0.01630652],
       [ 0.04040124, -0.02836997,  0.02959138, ...,  0.02061234,
        -0.061988  ,  0.0178513 ],
       [ 0.08200803, -0.04166204,  0.04411567, ...,  0.02821253,
        -0.08437772,  0.003652  ]], shape=(5, 384), dtype=float32)

In [28]:
kmeans.labels_

array([2, 2, 3, 3, 2, 0, 3, 2, 4, 2, 2, 2, 2, 1, 2, 1, 4, 3, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1,
       3, 3, 1, 2, 2, 1, 3], dtype=int32)

In [29]:
from sklearn.metrics import pairwise_distances_argmin 

**Note:** pairwise_distances_argmin(X, Y) - This function computes for each row in X, the index of the row of Y which is closest.

In [30]:
closest_indices = pairwise_distances_argmin(kmeans.cluster_centers_, embeddings) #This gives indices of the 5 sentences closest to each cluster center!

sorted_indices = sorted(closest_indices)
sorted_indices
#closest_indices = list(set(closest_indices))....Consider adding this --> [since sometimes two clusters can choose same 
#sentences indices and then you will have 1 repeated sentence and only 4 unique sentences. Set makes sure each indices is unique.]

[np.int64(5), np.int64(19), np.int64(36), np.int64(37), np.int64(45)]

In [31]:
summary = ' '.join(sentences[i] for i in sorted_indices)

In [32]:
summary # Summary of only the first article {i.e. sample['sentences'].iloc[0]}.

"Be aware of what you can't control. Spending time with friends is a great way to relax. If you're leaving a party after hours of laughing, sharing your feelings, and listening to your friends, taking a 20-minute solo drive before going home can help you wind down. Reading is a fantastic way to relax, especially before bed. Light it with gentle lamplight or candles."

### BERT Summarizer Function:

In [33]:
def bert_summarizer(sentences):
    embeddings = model.encode(sentences)
    if len(sentences) >= 5:
        cluster = 5
    else:
        cluster = len(sentences)
    kmeans = KMeans(n_clusters=cluster, random_state=0, n_init='auto').fit(embeddings)
    closest_indices = pairwise_distances_argmin(kmeans.cluster_centers_, embeddings)
    sorted_indices = sorted(closest_indices)
    summary = ' '.join(sentences[i] for i in sorted_indices)
    return summary



In [34]:
sample['bert_summary'] = sample['sentences'].apply(bert_summarizer)

In [35]:
sample.head(2)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,lexrank_summary,lexrank_score,lexrank_rouge1_f,lexrank_rouge2_f,lexrank_rougeL_f,bert_summary
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,"[Meditating is a great way to relax your mind,...","When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627,Going out to the movies or watching a movie on...,{'rouge1': Score(precision=0.08074534161490683...,0.144444,0.078652,0.144444,Be aware of what you can't control. Spending t...
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"[The beauty industry is a huge one, and it’s e...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954,Before you begin your career as a beauty exper...,{'rouge1': Score(precision=0.17355371900826447...,0.234637,0.022599,0.134078,"This includes beauty treatments, massages, tan..."


### Evaluation:

In [36]:
from rouge_score import rouge_scorer

In [37]:
# Intialize the scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

In [38]:
scores = scorer.score(sample['headline'].iloc[0], sample['bert_summary'].iloc[0])
scores

{'rouge1': Score(precision=0.1791044776119403, recall=0.631578947368421, fmeasure=0.27906976744186046),
 'rouge2': Score(precision=0.06060606060606061, recall=0.2222222222222222, fmeasure=0.09523809523809523),
 'rougeL': Score(precision=0.13432835820895522, recall=0.47368421052631576, fmeasure=0.20930232558139533)}

In [39]:
sample['bert_score'] = sample.apply(lambda x: scorer.score(x['headline'], x['bert_summary']), axis=1)

In [40]:
sample.head(2)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,lexrank_summary,lexrank_score,lexrank_rouge1_f,lexrank_rouge2_f,lexrank_rougeL_f,bert_summary,bert_score
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,"[Meditating is a great way to relax your mind,...","When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627,Going out to the movies or watching a movie on...,{'rouge1': Score(precision=0.08074534161490683...,0.144444,0.078652,0.144444,Be aware of what you can't control. Spending t...,"{'rouge1': (0.1791044776119403, 0.631578947368..."
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"[The beauty industry is a huge one, and it’s e...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954,Before you begin your career as a beauty exper...,{'rouge1': Score(precision=0.17355371900826447...,0.234637,0.022599,0.134078,"This includes beauty treatments, massages, tan...","{'rouge1': (0.1794871794871795, 0.241379310344..."


In [41]:
sample['bert_score'].iloc[0]['rouge1'].fmeasure


0.27906976744186046

In [42]:
sample['bert_rouge1_f'] = sample['bert_score'].apply(lambda x : x['rouge1'].fmeasure)
sample['bert_rouge2_f'] = sample['bert_score'].apply(lambda x : x['rouge2'].fmeasure)
sample['bert_rougeL_f'] = sample['bert_score'].apply(lambda x : x['rougeL'].fmeasure)

In [43]:
sample.head(2)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,lexrank_summary,lexrank_score,lexrank_rouge1_f,lexrank_rouge2_f,lexrank_rougeL_f,bert_summary,bert_score,bert_rouge1_f,bert_rouge2_f,bert_rougeL_f
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,"[Meditating is a great way to relax your mind,...","When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627,Going out to the movies or watching a movie on...,{'rouge1': Score(precision=0.08074534161490683...,0.144444,0.078652,0.144444,Be aware of what you can't control. Spending t...,"{'rouge1': (0.1791044776119403, 0.631578947368...",0.279070,0.095238,0.209302
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"[The beauty industry is a huge one, and it’s e...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954,Before you begin your career as a beauty exper...,{'rouge1': Score(precision=0.17355371900826447...,0.234637,0.022599,0.134078,"This includes beauty treatments, massages, tan...","{'rouge1': (0.1794871794871795, 0.241379310344...",0.205882,0.044776,0.132353


In [44]:
print(sample[['bert_rouge1_f', 'bert_rouge2_f', 'bert_rougeL_f']].mean())

bert_rouge1_f    0.314798
bert_rouge2_f    0.072087
bert_rougeL_f    0.174436
dtype: float64


### BERT summarizer is complete. ✅


### Content Coverage:

**Content Coverage: Measure what fraction of words in ref summary appear in gen. summary.**   
--> Content coverage is closely related to ROUGE Recall  — high recall means good coverage.  
--> Use ROUGE-1 recall - It measures fraction of words in ref. summary which appear in gen. summary.   

Coverage Metric - ROUGE1  recall  

**Note:**  
*Bert Coverage is worst out of the three 3 methods.Why?*  
--> Because of the metric used for content coverage i.e. ROUGE1 recall, which is lexical metric—it counts exact, overlapping word n-grams.   
This favours TFIDF.   
--> BERT selects semantically diverse sentences, that mean the same thing as ref. summmary but those sentences use different words.  
-->ROUGE1 recall punishes this.Thus, resulting in BERT getting a low 'content coverage score'.

[bert_coverage       0.369990  
tfidf_coverage      0.532094  
lexrank_coverage    0.486582]  

In [45]:
sample.head(2)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,lexrank_summary,lexrank_score,lexrank_rouge1_f,lexrank_rouge2_f,lexrank_rougeL_f,bert_summary,bert_score,bert_rouge1_f,bert_rouge2_f,bert_rougeL_f
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,"[Meditating is a great way to relax your mind,...","When you watch a movie, try to clear your mind...",{'rouge1': Score(precision=0.07471264367816093...,0.134715,0.031414,0.103627,Going out to the movies or watching a movie on...,{'rouge1': Score(precision=0.08074534161490683...,0.144444,0.078652,0.144444,Be aware of what you can't control. Spending t...,"{'rouge1': (0.1791044776119403, 0.631578947368...",0.279070,0.095238,0.209302
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"[The beauty industry is a huge one, and it’s e...",In this vibrant industry where trends are cons...,{'rouge1': Score(precision=0.10344827586206896...,0.160920,0.030888,0.091954,Before you begin your career as a beauty exper...,{'rouge1': Score(precision=0.17355371900826447...,0.234637,0.022599,0.134078,"This includes beauty treatments, massages, tan...","{'rouge1': (0.1794871794871795, 0.241379310344...",0.205882,0.044776,0.132353


##### 1. BERT Coverage-----

In [46]:
sample['bert_score'].iloc[0]['rouge1'].recall


0.631578947368421

In [47]:
sample['bert_coverage'] = sample['bert_score'].apply(lambda x : x['rouge1'].recall)

**Note:**   
While saving dataframe from previous notebooks, the tfidf_score and lexrank_score columns got converted to string from dictonary.  
Need the scores to be in dictonary format to extract rogue1 score from it.

##### 2. TFIDF Coverage-----

In [48]:
sample['tfidf_score'] = sample.apply(lambda x : scorer.score(x['headline'], x['tfidf_summary']), axis=1) 
#Overwrites 'tfidf_score' to convert it back to dictionary.

In [49]:
sample['tfidf_score'].iloc[0]['rouge1'].recall

0.6842105263157895

In [50]:
sample['tfidf_coverage'] = sample['tfidf_score'].apply(lambda x : x['rouge1'].recall)

##### 3. LexRank Coverage-----

In [51]:
sample['lexrank_score'] = sample.apply(lambda x : scorer.score(x['headline'], x['lexrank_summary']), axis=1)

In [52]:
sample['lexrank_coverage'] = sample['lexrank_score'].apply(lambda x : x['rouge1'].recall)

In [53]:
sample[['bert_coverage', 'tfidf_coverage', 'lexrank_coverage']].mean() #Average Content coverage scores.

bert_coverage       0.369990
tfidf_coverage      0.532094
lexrank_coverage    0.486582
dtype: float64

### Redundancy:

Average pairwise cosine similarity  
For redundancy (using cosine similarity) ---
1. Take all sentences in the 'generated summary'.Generate sentence embeddings for sentences.
2. Compute pairwise cosine similarity between summary sentences.
3. Average those similarities — higher average = more redundant summary.


In [54]:
import nltk
nltk.download('punkt') #NLTK's Tokenizer.
nltk.download('stopwords')
nltk.download('punkt_tab')

import numpy as np

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [55]:
sample['bert_summary'].iloc[0]

"Be aware of what you can't control. Spending time with friends is a great way to relax. If you're leaving a party after hours of laughing, sharing your feelings, and listening to your friends, taking a 20-minute solo drive before going home can help you wind down. Reading is a fantastic way to relax, especially before bed. Light it with gentle lamplight or candles."

In [56]:
summary

"Be aware of what you can't control. Spending time with friends is a great way to relax. If you're leaving a party after hours of laughing, sharing your feelings, and listening to your friends, taking a 20-minute solo drive before going home can help you wind down. Reading is a fantastic way to relax, especially before bed. Light it with gentle lamplight or candles."

In [57]:
tokenized_summary = nltk.sent_tokenize(summary) # Summary is tokenized to avoid hitting the SBERT's 512 token limit. 
#Now each tokenized sentences (which is almost always less than 512 token) gets sent to SBERT model one after another for embedding generation.

In [58]:
len(tokenized_summary)

5

In [59]:
#Generate Sentence Embeddings:
embeddings1 = model.encode(tokenized_summary) 
embeddings1

array([[ 0.05665511, -0.04696403, -0.02227624, ..., -0.01044051,
        -0.0507655 ,  0.00654491],
       [ 0.02028532,  0.01875564,  0.03658789, ...,  0.07779712,
        -0.09307685,  0.04952397],
       [ 0.1404258 , -0.04037273,  0.04800164, ...,  0.07096997,
        -0.11473259,  0.01062538],
       [ 0.11074273, -0.02359821, -0.01033809, ...,  0.01434446,
        -0.04976725,  0.03925851],
       [ 0.05993503,  0.0433175 ,  0.06804778, ...,  0.02967818,
        -0.03687518,  0.01785759]], shape=(5, 384), dtype=float32)

In [60]:
from sklearn.metrics.pairwise import cosine_similarity

In [61]:
x = cosine_similarity(embeddings1) #Computes pairwise cosine similarity between summary sentences using their embeddings.
print('Shape:\n', x.shape)
print('Matrix:\n', x)


Shape:
 (5, 5)
Matrix:
 [[1.0000001  0.23458287 0.23614582 0.24422267 0.2180633 ]
 [0.23458287 1.0000005  0.44073546 0.52597713 0.17800182]
 [0.23614582 0.44073546 1.         0.28672916 0.12448768]
 [0.24422267 0.52597713 0.28672916 0.99999994 0.1703822 ]
 [0.2180633  0.17800182 0.12448768 0.1703822  0.9999999 ]]


Note:  
-->Each row or column in the matrix represents the similarity score wrt to another.  
-->The Diagonal is 1.0 (almost), since a sentence is perfectly similar to itself.  
-->We can see in the 1st row or 1st column, sentence-1 is most similar to sentence-4.  
-->1st row/1st column represent sentence1, 2nd row/2nd column represent sentence2 and so on.  

--> *np.triu(x, k=1)* returns the upper triangle of a matrix or 2D array, replacing everything else with zeros.  
x: 2-D Array    
k=1: tells the function to exclude diagonal i.e. make diagonal zero as well  

k=0[Default]:Keeps the main diagonal and everything above it.

<br>

-->Why np.triu?    
1. similarity matrix(x) has similarity score for each pair of sentences twice in the matrix.  
x[0][1] = 0.23458287  
x[1][0] = 0.23458287 (same thing!)
2. This means that upper and lower matrix are identcal.
3. np.triu() keeps the upper triangle of matrix, ensuring each pair is counted only once.


In [62]:
#Averaging the similarity scores.
upper = np.triu(x, k=1)
upper

array([[0.        , 0.23458287, 0.23614582, 0.24422267, 0.2180633 ],
       [0.        , 0.        , 0.44073546, 0.52597713, 0.17800182],
       [0.        , 0.        , 0.        , 0.28672916, 0.12448768],
       [0.        , 0.        , 0.        , 0.        , 0.1703822 ],
       [0.        , 0.        , 0.        , 0.        , 0.        ]],
      dtype=float32)

In [63]:
upper[upper > 0] #upper[upper > 0] filters out those zeros and computes the mean of only the real similarity values.

array([0.23458287, 0.23614582, 0.24422267, 0.2180633 , 0.44073546,
       0.52597713, 0.17800182, 0.28672916, 0.12448768, 0.1703822 ],
      dtype=float32)

In [64]:
redundancy = upper[upper > 0].mean()
#[upper > 0] - This is called Boolean Array(returns only those value in array where boolean statement is True).

redundancy # redundancy score for summary of the 1st article only.

np.float32(0.2659328)

 **Redundancy Function**

In [65]:
def calc_redundancy(summary):
    tokenized_summary = nltk.sent_tokenize(summary)
    embeddings = model.encode(tokenized_summary)
    x = cosine_similarity(embeddings)
    upper = np.triu(x, k=1)
    redundancy = upper[upper > 0].mean()
    return redundancy
    

In [66]:
sample.head(2)

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score,rouge1_f,rouge2_f,rougeL_f,...,lexrank_rouge2_f,lexrank_rougeL_f,bert_summary,bert_score,bert_rouge1_f,bert_rouge2_f,bert_rougeL_f,bert_coverage,tfidf_coverage,lexrank_coverage
0,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,"[Meditating is a great way to relax your mind,...","When you watch a movie, try to clear your mind...","{'rouge1': (0.07471264367816093, 0.68421052631...",0.134715,0.031414,0.103627,...,0.078652,0.144444,Be aware of what you can't control. Spending t...,"{'rouge1': (0.1791044776119403, 0.631578947368...",0.279070,0.095238,0.209302,0.631579,0.684211,0.684211
1,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"[The beauty industry is a huge one, and it’s e...",In this vibrant industry where trends are cons...,"{'rouge1': (0.10344827586206896, 0.36206896551...",0.160920,0.030888,0.091954,...,0.022599,0.134078,"This includes beauty treatments, massages, tan...","{'rouge1': (0.1794871794871795, 0.241379310344...",0.205882,0.044776,0.132353,0.241379,0.362069,0.362069


In [67]:
#TFIDF Redundancy
sample['tfidf_redundancy'] = sample['tfidf_summary'].apply(calc_redundancy)

In [68]:
#LexRank Redundancy
sample['lexrank_redundancy'] = sample['lexrank_summary'].apply(calc_redundancy)

In [69]:
#BERT Redundancy
sample['bert_redundancy'] = sample['bert_summary'].apply(calc_redundancy)

In [70]:
sample[['tfidf_redundancy', 'lexrank_redundancy', 'bert_redundancy']].mean()

tfidf_redundancy      0.399259
lexrank_redundancy    0.442467
bert_redundancy       0.324652
dtype: float32

In [71]:
#Saving Dataframe as pickle file.
#(Pickle file makes sure that Data Type of columns don't change upon saving and loading it in new notebook.)
sample.to_pickle('final_df.pkl') 

# EVALUATION DONE!!!✅
